## Loading the data

In [2]:
import pandas as pd
import numpy as np

In [3]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geoloc = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

### Completeness Check (Missing Values)

In [4]:
def missing_values_report(df, table_name):
    total = len(df)                                                 # total number of rows in the data
    missing_values = df.isnull().sum()                              # calculating missing values
    missing_per = ((missing_values/total)*100).round(2)             # calculating missing values in percentage

    report = pd.DataFrame({                                         # storing information in dataframe
        'Column': missing_values.index,
        'Missing Count' : missing_values.values,
        'Missing %' : missing_per.values,
        'Data Type' : df.dtypes.values
    })

    report = report[report['Missing Count'] > 0]                    # isolating the missing values data
    report['Severity'] = report['Missing %'].apply(                 # calculating severity
        lambda x: 'Critical' if x > 20
        else 'Moderate' if x > 5
        else 'Low'
    )
    print(f"\n                      =======Missing Value Report: {table_name} ======= ")
    print(report.to_string(index=False))
    return report


missing_values_report(orders, "orders")


                      =======Missing Value Report: orders ======= 
                       Column  Missing Count  Missing % Data Type Severity
            order_approved_at            160       0.16    object      Low
 order_delivered_carrier_date           1783       1.79    object      Low
order_delivered_customer_date           2965       2.98    object      Low


,Column,Missing Count,Missing %,Data Type,Severity
4,order_approved_at,160,0.16,object,Low
5,order_delivered_carrier_date,1783,1.79,object,Low
6,order_delivered_customer_date,2965,2.98,object,Low


### Duplicates Check

In [5]:
def duplicate_report(df, key_col, table_name):
    total = len(df)
    duplicate_rows = df.duplicated().sum()
    duplicate_keys = df[key_col].duplicated().sum()

    print(f"\n              ======Duplicate Report: {table_name}======")
    print(f"Total Rows              :{total}")
    print(f"Fully Duplicate Rows    :{duplicate_rows} ({round(duplicate_rows*100/total, 2)})")
        


    return duplicate_report

duplicate_report(orders, "order_id", "Orders")


              ======Duplicate Report: Orders======
Total Rows              :99441
Fully Duplicate Rows    :0 (0.0)


<function __main__.duplicate_report(df, key_col, table_name)>

## Validity Check (Format & Range)

In [6]:
def validity_report(df, table_name):
    print(f"\n          ======Validity Report: {table_name} =   ====")

    # date columns - are they parsable?
    date_cols =[col for col in df.columns if'date' in col or 'timestamp' in col]
    
    for col in date_cols:
        try:
            converted_cols = pd.date_time(df[col],errors = 'coerce')
            failed = converted_cols.isnull().sum() - df[col].isnull().sum()
            print(f"Date column '{col}' : {failed} unparseable values")
        except:
            print(f"Date column '{col}' : Could not convert")

    # Numberic columns - check for negatives or outliers

    num_cols = df.select_dtypes(include = [np.number]).columns
    for col in num_cols:
        negatives = (df[col] < 0).sum()
        if negatives > 0:
            print(f"Numeric columns '{col}': {negatives} negative values")
    

validity_report(orders, "Orders")


          ======Validity Report: Orders =   ====
Date column 'order_purchase_timestamp' : Could not convert
Date column 'order_delivered_carrier_date' : Could not convert
Date column 'order_delivered_customer_date' : Could not convert
Date column 'order_estimated_delivery_date' : Could not convert


## Summary Statistics

In [7]:
def summary_stats_report(df, table_name):
    print(f"            ===== Summary Statistics: {table_name}=====")
    print(df.describe(include='all').T)

summary_stats_report(orders, "Orders")

            ===== Summary Statistics: Orders=====
                               count unique                               top  \
order_id                       99441  99441  e481f51cbdc54678b7cc49136f2d6af7   
customer_id                    99441  99441  9ef432eb6251297304e76186b10a928d   
order_status                   99441      8                         delivered   
order_purchase_timestamp       99441  98875               2018-08-02 12:05:26   
order_approved_at              99281  90733               2018-02-27 04:31:10   
order_delivered_carrier_date   97658  81018               2018-05-09 15:48:00   
order_delivered_customer_date  96476  95664               2018-05-08 19:36:48   
order_estimated_delivery_date  99441    459               2017-12-20 00:00:00   

                                freq  
order_id                           1  
customer_id                        1  
order_status                   96478  
order_purchase_timestamp           3  
order_approved_at        

## FULL DATA QUALITY REPORT

In [8]:
def full_data_quality_report(dataframes: dict):
    """
        dataframes: dictionary of {table name: dataframe}
        key_columns: primary key per table
    """

    key_map = {
        'orders':'order_id',
        'customers':'customer_id',
        'products':'product_id',
        'sellers':'seller_id'
    }

    all_missing = []

    for name, df in dataframes.items():
        print(f"\n{'='*50}")
        print(f"TABLE: {name.upper()}")
        print(f"{'='*50}")
        print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

        m = missing_values_report(df, name)
        all_missing.append(m)

        if name in key_map:
            duplicate_report(df, key_map[name], name)

        validity_report(df, name)

dataframes = {
    'orders' : orders,
    'customers' : customers,
    'products' : products,
    'geoloc' : geoloc, 
    'order_items' : order_items,
    'order_payments' : order_payments,
    'order_reviews' : order_reviews,
    'sellers' : sellers
}

full_data_quality_report(dataframes)




TABLE: ORDERS
Rows: 99441 | Columns: 8

                      =======Missing Value Report: orders ======= 
                       Column  Missing Count  Missing % Data Type Severity
            order_approved_at            160       0.16    object      Low
 order_delivered_carrier_date           1783       1.79    object      Low
order_delivered_customer_date           2965       2.98    object      Low

              ======Duplicate Report: orders======
Total Rows              :99441
Fully Duplicate Rows    :0 (0.0)

          ======Validity Report: orders =   ====
Date column 'order_purchase_timestamp' : Could not convert
Date column 'order_delivered_carrier_date' : Could not convert
Date column 'order_delivered_customer_date' : Could not convert
Date column 'order_estimated_delivery_date' : Could not convert

TABLE: CUSTOMERS
Rows: 99441 | Columns: 5

                      =======Missing Value Report: customers ======= 
Empty DataFrame
Columns: [Column, Missing Count, Missing %, Da

## Validating report discrepanices

### ORDERS DATA

##### Orders table shows low severity missing values for order_approved_at, order_delivered_carrier_date & order_delivered_customer_date

In [16]:
orders[orders['order_approved_at'].isnull()].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaN,NaN,NaN,2017-04-10 00:00:00
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaN,NaN,NaN,2018-09-13 00:00:00
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaN,NaN,NaN,2017-05-30 00:00:00


In [17]:
# Let's check the order status for these orders
orders[orders['order_approved_at'].isnull()]['order_status'].unique()

array(['canceled', 'delivered', 'created'], dtype=object)

##### There are some orders that were fullfilled.

In [35]:
# Let's check the details about the orderes that were delivered
orders[orders['order_approved_at'].isnull()][orders['order_status'] == 'delivered'].head()

C:\Users\Tejali\AppData\Local\Temp\ipykernel_20048\2206964985.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  orders[orders['order_approved_at'].isnull()][orders['order_status'] == 'delivered'].head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00


In [34]:
# Calculating the % of delivered orders with no order approval timestamp
round(orders[orders['order_approved_at'].isnull()][orders['order_status'] == 'delivered'].shape[0]/orders[orders['order_status'] == 'delivered'].shape[0], 4)

C:\Users\Tejali\AppData\Local\Temp\ipykernel_20048\3349783064.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  round(orders[orders['order_approved_at'].isnull()][orders['order_status'] == 'delivered'].shape[0]/orders[orders['order_status'] == 'delivered'].shape[0], 4)


0.0001

In [50]:
# Let's check the different status options available for each order
orders['order_status'].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

#### Given the missing data for specific columns - comparing for which of them it is normal to have missing data for each type of order status - order_approved_at	order_delivered_carrier_date	order_delivered_customer_date

'created' - order_approved_at | order_delivered_carrier_date | order_delivered_customer_date

'canceled' - order_approved_at | order_delivered_carrier_date | order_delivered_customer_date

'unavailable' - order_approved_at | order_delivered_carrier_date | order_delivered_customer_date

'approved' - order_delivered_carrier_date | order_delivered_customer_date

'processing' - order_delivered_carrier_date | order_delivered_customer_date

'invoiced' -  order_delivered_carrier_date | order_delivered_customer_date

'shipped' - order_delivered_carrier_date | order_delivered_customer_date

'delivered' - all missing columns information expected to be present





 


#### Orders delivered before purchase data

In [55]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].unique()

array(['invoiced', 'shipped', 'processing', 'unavailable', 'canceled',
       'delivered', 'created', 'approved'], dtype=object)

### PRODUCTS DATA